# TUGAS MANDIRI PRAKTIKUM BIG DATA — PERTEMUAN 6
## Format Data Besar (Parquet & JSON) dan Pipeline ETL Sederhana

| Informasi | Detail |
|---|---|
| **Mata Kuliah** | Praktikum Big Data |
| **Pertemuan** | 6 (Tugas Mandiri) |
| **Nama Mahasiswa** | Yarsya |
| **NPM** | 2505060066 |
| **Program Studi** | S1 Teknologi Informasi — Universitas Tidar |

---

### Skenario Tugas
Membangun pipeline ETL (*Extract → Transform → Load*) produksi tingkat awal yang mengintegrasikan tiga sumber data heterogen: data transaksi (CSV), data katalog produk (JSON Lines), dan data ulasan pelanggan (CSV). Data diekstrak langsung dari HDFS, ditransformasikan dengan join relasional bertingkat (*left join* dan *inner join*), dibersihkan dan diperkaya (*data quality & feature engineering*), dimuat kembali ke HDFS dalam format kolumnar **Parquet terpartisi**, serta dianalisis untuk menghasilkan wawasan bisnis bagi tim marketing.

## 0. Inisialisasi SparkSession
Menyiapkan lingkungan PySpark dengan konfigurasi path python yang konsisten.

In [1]:
import os
import sys

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, count, round

spark = SparkSession.builder \
    .appName("Tugas6-ETL-Pipeline-Yarsya") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("SparkSession berhasil diinisialisasi: Versi", spark.version)

26/11/05 17:15:36 WARN Utils: Your hostname, meil resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/11/05 17:15:36 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/05 17:15:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession berhasil diinisialisasi: Versi 3.5.9


## 1. Menyiapkan Tiga Sumber Data & Unggah ke HDFS
Membangkitkan data transaksi (5.000 baris), katalog produk (30 produk dalam format JSON Lines), dan ulasan pelanggan (3.500 baris) lalu menyimpannya ke HDFS.

In [2]:
import numpy as np
import pandas as pd
import json

np.random.seed(33)

# Sumber 1: Transaksi (CSV) — 5000 baris
n_trx = 5000
kategori_list = ["Elektronik", "Fashion", "Makanan", "Rumah Tangga", "Kesehatan"]
df_trx_local = pd.DataFrame({
    "order_id": [f"TX{i}" for i in range(n_trx)],
    "product_id": np.random.randint(1, 31, size=n_trx),
    "unit_terjual": np.random.randint(1, 8, size=n_trx),
    "tanggal": np.random.choice(pd.date_range("2026-10-01","2026-10-31"), size=n_trx).astype(str),
})
df_trx_local.to_csv("tugas6_transaksi.csv", index=False)

# Sumber 2: Data produk (JSON Lines) — 30 produk
produk = [
    {"product_id": i, "nama_produk": f"Produk-{i}", "kategori": np.random.choice(kategori_list),
     "harga": int(np.random.choice([25000, 50000, 75000, 100000, 150000, 250000]))}
    for i in range(1, 31)
]
with open("tugas6_produk.json", "w") as f:
    for p in produk:
        f.write(json.dumps(p) + "\n")

# Sumber 3: Data ulasan (CSV) — 3500 ulasan
n_review = 3500
df_review_local = pd.DataFrame({
    "order_id": np.random.choice(df_trx_local["order_id"], size=n_review, replace=False),
    "rating": np.random.randint(1, 6, size=n_review),
})
df_review_local.to_csv("tugas6_ulasan.csv", index=False)

# Unggah ke HDFS raw directory
!hdfs dfs -mkdir -p /user/mahasiswa/tugas6/raw
!hdfs dfs -put -f tugas6_transaksi.csv tugas6_produk.json tugas6_ulasan.csv /user/mahasiswa/tugas6/raw/
!hdfs dfs -ls -h /user/mahasiswa/tugas6/raw

Found 3 items
-rw-r--r--   1 meil supergroup      2.6 K 2026-11-05 17:15 /user/mahasiswa/tugas6/raw/tugas6_produk.json
-rw-r--r--   1 meil supergroup    202.6 K 2026-11-05 17:15 /user/mahasiswa/tugas6/raw/tugas6_transaksi.csv
-rw-r--r--   1 meil supergroup     30.0 K 2026-11-05 17:15 /user/mahasiswa/tugas6/raw/tugas6_ulasan.csv


---
## Bagian A. EXTRACT (Bobot 15%)

Membaca ketiga sumber data heterogen dari HDFS ke dalam tiga Spark DataFrame terpisah (`df_transaksi`, `df_produk`, `df_ulasan`), lalu menampilkan skema kolom dan total baris masing-masing.

In [3]:
raw_base_path = "hdfs://localhost:9000/user/mahasiswa/tugas6/raw"

# Membaca ketiga sumber data dari HDFS
df_transaksi = spark.read.csv(f"{raw_base_path}/tugas6_transaksi.csv", header=True, inferSchema=True)
df_produk = spark.read.json(f"{raw_base_path}/tugas6_produk.json")
df_ulasan = spark.read.csv(f"{raw_base_path}/tugas6_ulasan.csv", header=True, inferSchema=True)

# Menampilkan skema dan jumlah baris masing-masing sumber
print(f"1. df_transaksi : {df_transaksi.count()} baris")
df_transaksi.printSchema()

print(f"\n2. df_produk : {df_produk.count()} baris")
df_produk.printSchema()

print(f"\n3. df_ulasan : {df_ulasan.count()} baris")
df_ulasan.printSchema()

1. df_transaksi : 5000 baris
root
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)


2. df_produk : 30 baris
root
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- product_id: long (nullable = true)




3. df_ulasan : 3500 baris
root
 |-- order_id: string (nullable = true)
 |-- rating: integer (nullable = true)



---
## Bagian B. TRANSFORM — Penggabungan (Bobot 25%)

Menggabungkan ketiga DataFrame menjadi satu:
- **Left Join** dari `df_transaksi` ke `df_ulasan` (kunci: `order_id`): Karena tidak seluruh transaksi memiliki ulasan, *left join* memastikan transaksi tanpa ulasan tidak terbuang.
- **Inner Join** dari hasil gabungan ke `df_produk` (kunci: `product_id`): Karena seluruh transaksi memiliki produk yang valid dalam master katalog.
- Menambahkan kolom kalkulasi **`total_pendapatan`** = `unit_terjual * harga`.

In [4]:
# 1. Left join transaksi ke ulasan
df_trx_review = df_transaksi.join(df_ulasan, on="order_id", how="left")

# 2. Inner join ke produk
df_gabungan = df_trx_review.join(df_produk, on="product_id", how="inner") \
    .withColumn("total_pendapatan", col("unit_terjual") * col("harga"))

print(f"Total baris setelah penggabungan: {df_gabungan.count()} baris")
df_gabungan.select("order_id", "product_id", "nama_produk", "kategori", "unit_terjual", "harga", "total_pendapatan", "rating").show(10)

Total baris setelah penggabungan: 5000 baris


+--------+----------+-----------+------------+------------+------+----------------+------+
|order_id|product_id|nama_produk|    kategori|unit_terjual| harga|total_pendapatan|rating|
+--------+----------+-----------+------------+------------+------+----------------+------+
|     TX0|        21|  Produk-21|  Elektronik|           2| 25000|           50000|     1|
|     TX1|         8|   Produk-8|     Fashion|           6|250000|         1500000|  NULL|
|     TX2|        25|  Produk-25|  Elektronik|           4| 25000|          100000|     2|
|     TX3|         3|   Produk-3|     Fashion|           4| 75000|          300000|     5|
|     TX4|        19|  Produk-19|Rumah Tangga|           6| 75000|          450000|  NULL|
|     TX5|        10|  Produk-10|  Elektronik|           3| 75000|          225000|     5|
|     TX6|        26|  Produk-26|   Kesehatan|           6| 75000|          450000|     4|
|     TX7|         4|   Produk-4|     Makanan|           1|150000|          150000|  NULL|

---
## Bagian C. TRANSFORM — Penanganan Data Kosong & Pengayaan (Bobot 20%)

1. Menambahkan kolom boolean **`ada_ulasan`** (`True`/`False`) berdasarkan kondisi `col("rating").isNotNull()`, dieksekusi **sebelum** penanganan `null`.
2. Mengisi nilai kosong pada kolom `rating` dengan angka `0` menggunakan `df.na.fill({"rating": 0})`.

In [5]:
# 1. Menambahkan kolom boolean ada_ulasan sebelum na.fill()
df_enriched = df_gabungan.withColumn("ada_ulasan", col("rating").isNotNull())

# 2. Mengisi rating null dengan angka 0
df_final = df_enriched.na.fill({"rating": 0})

print("Sampel Data Hasil Transformasi (menampilkan perbandingan ada_ulasan True dan False):")
df_final.select("order_id", "nama_produk", "kategori", "unit_terjual", "total_pendapatan", "rating", "ada_ulasan").show(10)

Sampel Data Hasil Transformasi (menampilkan perbandingan ada_ulasan True dan False):


+--------+-----------+------------+------------+----------------+------+----------+
|order_id|nama_produk|    kategori|unit_terjual|total_pendapatan|rating|ada_ulasan|
+--------+-----------+------------+------------+----------------+------+----------+
|     TX0|  Produk-21|  Elektronik|           2|           50000|     1|      true|
|     TX1|   Produk-8|     Fashion|           6|         1500000|     0|     false|
|     TX2|  Produk-25|  Elektronik|           4|          100000|     2|      true|
|     TX3|   Produk-3|     Fashion|           4|          300000|     5|      true|
|     TX4|  Produk-19|Rumah Tangga|           6|          450000|     0|     false|
|     TX5|  Produk-10|  Elektronik|           3|          225000|     5|      true|
|     TX6|  Produk-26|   Kesehatan|           6|          450000|     4|      true|
|     TX7|   Produk-4|     Makanan|           1|          150000|     0|     false|
|     TX8|   Produk-7|  Elektronik|           2|          150000|     5|    

### Rasionalisasi Pengisian Rating Kosong dengan Angka 0
Pengisian nilai `null` pada kolom rating dengan angka **0** (bukan rata-rata seperti 3 atau nilai lainnya) memiliki landasan bisnis yang logis:
1. **Pemisahan Semantik Ulasan:** Skala rating ulasan pelanggan berada pada rentang 1 hingga 5 bintang. Angka 0 bertindak sebagai nilai khusus (*sentinel value*) yang merepresentasikan **"ketiadaan ulasan"** (belum diulas), bukan ulasan buruk (bintang 1) ataupun ulasan netral (bintang 3).
2. **Integritas Metrik Finansial:** Pendekatan ini mempertahankan seluruh transaksi penjualan tanpa kehilangan baris transaksi (*no data loss*), sekaligus memudahkan filtering analitik kepuasan (`WHERE rating > 0`) tanpa merusak data penjualan.

---
## Bagian D. LOAD (Bobot 25%)

Menyimpan hasil akhir ke HDFS dalam format kolumnar terkompresi **Parquet**, dipartisi berdasarkan kolom `kategori` (`partitionBy("kategori")`), ke path `/user/mahasiswa/tugas6/hasil_etl`.

Setelah itu, dilakukan verifikasi struktur partisi folder menggunakan `hdfs dfs -ls -R` dan pembacaan kembali untuk membuktikan integritas jumlah baris.

In [6]:
# Menyimpan ke HDFS format Parquet terpartisi berdasarkan kategori
output_parquet_path = "hdfs://localhost:9000/user/mahasiswa/tugas6/hasil_etl"
df_final.write.mode("overwrite").partitionBy("kategori").parquet(output_parquet_path)

print("Data hasil ETL berhasil disimpan ke HDFS dalam format Parquet terpartisi!")

Data hasil ETL berhasil disimpan ke HDFS dalam format Parquet terpartisi!


In [7]:
# Verifikasi struktur direktori partisi di HDFS
!hdfs dfs -ls -R /user/mahasiswa/tugas6/hasil_etl

-rw-r--r--   3 meil supergroup          0 2026-11-05 17:15 /user/mahasiswa/tugas6/hasil_etl/_SUCCESS
drwxr-xr-x   - meil supergroup          0 2026-11-05 17:15 /user/mahasiswa/tugas6/hasil_etl/kategori=Elektronik
-rw-r--r--   3 meil supergroup      16366 2026-11-05 17:15 /user/mahasiswa/tugas6/hasil_etl/kategori=Elektronik/part-00000-94d05f86-ca10-4a59-9942-b920bde1ecf9.c000.snappy.parquet
drwxr-xr-x   - meil supergroup          0 2026-11-05 17:15 /user/mahasiswa/tugas6/hasil_etl/kategori=Fashion
-rw-r--r--   3 meil supergroup      11133 2026-11-05 17:15 /user/mahasiswa/tugas6/hasil_etl/kategori=Fashion/part-00000-94d05f86-ca10-4a59-9942-b920bde1ecf9.c000.snappy.parquet
drwxr-xr-x   - meil supergroup          0 2026-11-05 17:15 /user/mahasiswa/tugas6/hasil_etl/kategori=Kesehatan
-rw-r--r--   3 meil supergroup      10392 2026-11-05 17:15 /user/mahasiswa/tugas6/hasil_etl/kategori=Kesehatan/part-00000-94d05f86-ca10-4a59-9942-b920bde1ecf9.c000.snappy.parquet
drwxr-xr-x   - meil supergroup 

-rw-r--r--   3 meil supergroup       9269 2026-11-05 17:15 /user/mahasiswa/tugas6/hasil_etl/kategori=Rumah Tangga/part-00000-94d05f86-ca10-4a59-9942-b920bde1ecf9.c000.snappy.parquet


In [8]:
# Verifikasi membaca kembali data Parquet dari HDFS dan memeriksa integritas baris
df_read_parquet = spark.read.parquet(output_parquet_path)
total_baris_baca = df_read_parquet.count()

print(f"Total baris setelah dibaca kembali dari Parquet HDFS: {total_baris_baca} baris")
assert total_baris_baca == 5000, "Jumlah baris tidak sesuai!"
print("Verifikasi Integritas Data: SUKSES (100% data 5000 baris terjaga utuh).")

Total baris setelah dibaca kembali dari Parquet HDFS: 5000 baris
Verifikasi Integritas Data: SUKSES (100% data 5000 baris terjaga utuh).


---
## Bagian E. Insight Akhir (Bobot 15%)

Menghitung persentase transaksi yang memiliki ulasan (`ada_ulasan = True`) per kategori produk untuk mengidentifikasi kategori dengan tingkat partisipasi ulasan terendah.

In [9]:
# Menghitung persentase transaksi dengan ulasan per kategori
df_persen_ulasan = df_final.groupBy("kategori") \
    .agg(
        count("order_id").alias("total_transaksi"),
        count(when(col("ada_ulasan") == True, True)).alias("transaksi_ada_ulasan"),
        round((count(when(col("ada_ulasan") == True, True)) / count("order_id")) * 100, 2).alias("persentase_berulasan")
    ) \
    .orderBy(col("persentase_berulasan").asc())

print("Tabel Persentase Transaksi dengan Ulasan per Kategori Produk:")
df_persen_ulasan.show()

kategori_terendah = df_persen_ulasan.first()
print(f"-> Kategori dengan persentase ulasan paling rendah adalah '{kategori_terendah['kategori']}' ({kategori_terendah['persentase_berulasan']} %)")

Tabel Persentase Transaksi dengan Ulasan per Kategori Produk:


+------------+---------------+--------------------+--------------------+
|    kategori|total_transaksi|transaksi_ada_ulasan|persentase_berulasan|
+------------+---------------+--------------------+--------------------+
|     Makanan|            536|                 369|               68.84|
|   Kesehatan|            961|                 662|               68.89|
|     Fashion|           1035|                 726|               70.14|
|Rumah Tangga|            815|                 575|               70.55|
|  Elektronik|           1653|                1168|               70.66|
+------------+---------------+--------------------+--------------------+



-> Kategori dengan persentase ulasan paling rendah adalah 'Makanan' (68.84 %)


### Interpretasi Bisnis untuk Tim Marketing

Temuan bahwa kategori dengan tingkat ulasan terendah (seperti terlihat pada tabel di atas) sangat penting untuk ditindaklanjuti oleh tim marketing karena ulasan (*customer reviews*) merupakan faktor pendorong konversi (*social proof*) utama dalam keputusan pembelian konsumen e-commerce. Rendahnya partisipasi ulasan pada kategori tersebut mengindikasikan minimnya interaksi pascapembelian, sehingga tim marketing perlu merancang program insentif ulasan—seperti *loyalty coins*, kupon diskon pesanan berikutnya, atau notifikasi pengingat interaktif di aplikasi—khusus bagi pelanggan yang membeli produk dari kategori ini guna memperkuat reputasi produk dan mendongkrak penjualan jangka panjang.